<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec05_neighbor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec05_neighbor.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第5回 近傍処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec05_neighbor.m` を Python に移植したもの。第5回のスライド（vie2026-05）の図と数値を計算する。線形フィルタ（平均値・加重平均値・ラプラシアン・アンシャープマスク），中央値フィルタ，勾配フィルタ（プレウィット・ソーベル），線形シフト不変システムと畳み込みを扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

教科書の線形フィルタは局所的な積和演算

$$y[\boldsymbol{n}] = \sum_{\boldsymbol{m}\in\mathcal{N}_\mathrm{f}} f[\boldsymbol{m}]\, x[\boldsymbol{n}+\boldsymbol{m}]$$

で表される（実数の場合）。SciPy の `scipy.ndimage.correlate` はこの形（相関）を計算する。`mode="constant"` とすれば周囲を零値で拡張する。

教科書（3.1〜3.3 節）の例題は，すべて同じ $3\times 4$ 配列 $\mathsf{x}$ を使う。画像処理には SciPy，scikit-image，NumPy を使う。各節で計算結果を教科書の解答と照合し（`assert`），スライドの数値例（配列の値，途中の式，答え）と一致することを確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。図の配色はロゴの色（メインの緑，暖色系の橙）に合わせる。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.util import img_as_float, random_noise
from skimage.metrics import peak_signal_noise_ratio
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

途中の式を文字列にする関数を定義する（負の値は括弧で囲む）。

In [ ]:
def numtex(v):
    """数値を LaTeX の文字列にする（負の値は括弧で囲む）．"""
    s = f"{v:g}"
    return f"({s})" if v < 0 else s


def sumexpr(v):
    """数値の並びを "a+b+c" の形の式にする（負の値は括弧で囲む）．"""
    return "+".join(numtex(x) for x in np.ravel(v))


def prodexpr(f, w):
    """係数 f と画素値 w の積和を行ごとに "f1\\cdot w1+f2\\cdot w2+..." の形の式にする．"""
    s = ""
    for k, (fk, wk) in enumerate(zip(np.ravel(f), np.ravel(w))):
        t = f"{abs(fk):g}\\cdot" + numtex(wk)
        s += ("-" + t) if fk < 0 else ("+" + t if k > 0 else t)
    return s


def ordlist(v, K):
    """昇順に並べた値を "a,b,\\underline{c},..." の形にする（K 番目に下線．K は 1 始まり）．"""
    t = [f"{x:g}" for x in v]
    t[K - 1] = "\\underline{" + t[K - 1] + "}"
    return ",".join(t)

## 数値例の配列
教科書の例題で使う $3\times 4$ 配列。周囲の値はすべて零値と仮定する。

In [ ]:
X = np.array([[18, 9, 9, 9], [27, 9, 9, 9], [36, 9, 9, 9]], dtype=float)
print(X.astype(int))

周囲の零値拡張 $\mathsf{x}_\mathrm{pad}$（`np.pad`）。スライドでは枠の外側の零値を灰色で示す。

In [ ]:
Xpad = np.pad(X, 1)                            # 上下左右に 1 画素ずつ零値を付ける
print(Xpad.astype(int))

出力画素 $y[n_\mathrm{v},n_\mathrm{h}]$ が参照する $3\times 3$ の局所領域を取り出す関数。$\mathsf{x}_\mathrm{pad}$ では行・列の番号が 1 ずれるので，$\mathsf{x}_\mathrm{pad}$ の $n_\mathrm{v}$〜$n_\mathrm{v}+2$ 行，$n_\mathrm{h}$〜$n_\mathrm{h}+2$ 列を取り出せばよい。

In [ ]:
def win(nv, nh):
    return Xpad[nv:nv + 3, nh:nh + 3]


print(win(1, 1).astype(int))                   # y[1,1] の局所領域

## 平均値（矩形）フィルタ
$3\times 3$ 矩形フィルタのカーネルは $\frac{1}{9}\mathbf{1}$。出力は 9 画素の平均（教科書 例題「矩形フィルタ」）。

In [ ]:
fbox = np.ones((3, 3)) / 9
Ybox = ndimage.correlate(X, fbox, mode="constant")   # 周囲は零値
YboxBook = np.array([[7, 9, 6, 4], [12, 15, 9, 6], [9, 11, 6, 4]])   # 教科書の解答
assert np.max(np.abs(Ybox - YboxBook)) < 1e-12, "矩形フィルタ：教科書の解答と一致しない"
Ybox = np.round(Ybox).astype(int)              # 丸め誤差を除いて整数にする
print(Ybox)

教科書の解答にならって，局所領域の画素値を行ごとに左から足し，9 で割る式を作る。$y[1,1]$（スライドで強調する画素）と，零値拡張の効く $y[0,0]$，$y[1,0]$。

In [ ]:
boxExpr = {}
for nv, nh in [(1, 1), (0, 0), (1, 0)]:
    e = "(" + sumexpr(win(nv, nh)) + ")/9"
    boxExpr[(nv, nh)] = e
    print(f"y[{nv},{nh}] = {e} = {Ybox[nv, nh]}")
# スライドの式と一致することを確かめる
assert boxExpr[(1, 1)] == "(18+9+9+27+9+9+36+9+9)/9"
assert boxExpr[(0, 0)] == "(0+0+0+0+18+9+0+27+9)/9"
assert boxExpr[(1, 0)] == "(0+18+9+0+27+9+0+36+9)/9"

## 加重平均値（ガウシアン）フィルタ
教科書の式 (3.3)(3.4) どおりに，距離の 2 乗 $\|\boldsymbol{m}\|_2^2=m_\mathrm{v}^2+m_\mathrm{h}^2$ からガウス関数 $g[\boldsymbol{m}]$ を求め，総和で割って正規化する。

In [ ]:
mh, mv = np.meshgrid(np.arange(-1, 2), np.arange(-1, 2))   # m = (m_v, m_h)，行が m_v，列が m_h


def gauss(sg):
    g = np.exp(-(mv**2 + mh**2) / (2 * sg**2))
    return g / g.sum()

教科書 例「ガウシアンフィルタ」：$\sigma_\mathrm{g}=0.5$。中央の重みが大きく，平滑化は弱い。

In [ ]:
fg05 = gauss(0.5)
np.set_printoptions(precision=4, suppress=True)
print(fg05)
fg05Book = np.array([[0.0113, 0.0838, 0.0113], [0.0838, 0.6193, 0.0838], [0.0113, 0.0838, 0.0113]])
assert np.max(np.abs(fg05 - fg05Book)) < 5e-5, "ガウシアン（σ=0.5）：教科書の値と一致しない"

教科書 例題「ガウシアンフィルタ」：$\sigma_\mathrm{g}\simeq 0.849$ のカーネルは $\frac{1}{16}\begin{pmatrix}1&2&1\\2&4&2\\1&2&1\end{pmatrix}$ にほぼ等しい。SciPy の `gaussian_filter`（半径 1 で打ち切り）でインパルスを平滑化した結果とも一致する。

In [ ]:
fg = gauss(0.849)
print(fg)
fgBook = np.array([[0.0625, 0.1250, 0.0625], [0.1250, 0.2501, 0.1250], [0.0625, 0.1250, 0.0625]])
assert np.max(np.abs(fg - fgBook)) < 5e-5, "ガウシアン（σ=0.849）：教科書の値と一致しない"
delta = np.zeros((3, 3))
delta[1, 1] = 1
assert np.max(np.abs(fg - ndimage.gaussian_filter(delta, 0.849, mode="constant", truncate=1 / 0.849))) < 1e-12
fg16 = np.round(16 * fg).astype(int)           # 16 倍して丸めると整数のカーネル
print(fg16)
assert np.array_equal(fg16, [[1, 2, 1], [2, 4, 2], [1, 2, 1]])

## フィルタカーネルの形状
講義のデモ（linearfilter.m）にならい，重み係数 $f[\boldsymbol{m}]$ を立体的な棒グラフ（`stem`）で示す。矩形フィルタは平ら，加重平均値フィルタは中央が高い。比べやすいよう縦軸をそろえる。

In [ ]:
fig = plt.figure(figsize=(8, 3.6), layout="constrained")
for k, (krn, ttl) in enumerate([(fbox, "矩形フィルタ"), (fg16 / 16, "加重平均値フィルタ")]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ml, sl, bl = ax.stem(mh.ravel(), mv.ravel(), krn.ravel(), basefmt=" ")
    plt.setp(ml, color=cmain, markersize=4)
    plt.setp(sl, color=cmain, linewidth=1.2)
    ax.set(xlabel="$m_\\mathrm{h}$", ylabel="$m_\\mathrm{v}$", xticks=[-1, 0, 1], yticks=[-1, 0, 1],
           zlim=(0, 0.3), zticks=np.arange(0, 0.31, 0.1), ylim=(1.2, -1.2), title=ttl)
    ax.view_init(elev=30, azim=-125)
plt.show()

## 平滑化の処理例（ガウス性ノイズ）
参考資料のサンプル画像 msipimg08（スイカ）をグレースケールにして $256\times 256$ 画素に縮小し，標準偏差 0.05 のガウス性ノイズを加え，平均値フィルタと加重平均値フィルタで平滑化する。暗い背景や縞の内側など平坦な部分が多く，ノイズの除去が目で見ても PSNR でもはっきり分かる（細かな模様の多い画像では平滑化によるボケの損失が勝ち，PSNR がかえって下がる）。雑音は scikit-image の `random_noise` で加える。乱数と縮小の実装が MATLAB と異なるため，PSNR の値はスライドとわずかに異なる。

In [ ]:
C = img_as_float(vie.msipimg(8, 256, "gray"))
Cg = random_noise(C, mode="gaussian", mean=0, var=0.05**2, rng=0)   # 乱数の種を固定して再現性を保つ
Cbox = ndimage.correlate(Cg, fbox, mode="nearest")                  # 周囲は端の値で拡張
Cgau = ndimage.correlate(Cg, fg16 / 16, mode="nearest")
psnrG = [peak_signal_noise_ratio(C, v, data_range=1) for v in (Cg, Cbox, Cgau)]
print("psnrG =", np.round(psnrG, 1), "dB")
fig, axs = plt.subplots(2, 2, figsize=(7, 7.4), layout="constrained")
show(axs[0, 0], C, "原画像")
show(axs[0, 1], Cg, "雑音重畳（ガウス性）")
show(axs[1, 0], Cbox, "平均値フィルタ")
show(axs[1, 1], Cgau, "加重平均値フィルタ")
plt.show()

## 中央値フィルタ
$3\times 3$ 領域の画素値を昇順に並べ，5 番目（中央，$K=(9+1)/2$）の値で置き換える。外れ値 100 の影響を受けない。

In [ ]:
B = np.array([[10, 20, 20], [20, 15, 20], [20, 25, 100]])
print(B)
srt = np.sort(B.ravel())
K = (B.size + 1) // 2
med = srt[K - 1]                               # K 番目（Python の添字は 0 始まり）
assert med == np.median(B)
avg = B.mean()                                 # 平均値は外れ値に引っ張られる
print("sorted =", srt, " K =", K, " med =", med)
print(f"和 = {B.sum()}，平均 = {avg:.1f}")
print(ordlist(srt, K))                         # K 番目に下線
assert ordlist(srt, K) == "10,15,20,20,\\underline{20},20,20,25,100" and B.sum() == 250

数値例の配列 $\mathsf{x}$ に対する $3\times3$ 中央値フィルタ（周囲は零値，教科書 例題「中央値フィルタ」）。順序統計フィルタ `rank_filter` で $K=5$（0 始まりの順位 4）としても同じ。

In [ ]:
Ymed = ndimage.median_filter(X, size=3, mode="constant", cval=0)
assert np.array_equal(Ymed, ndimage.rank_filter(X, rank=4, size=3, mode="constant", cval=0))
YmedBook = np.array([[0, 9, 9, 0], [9, 9, 9, 9], [0, 9, 9, 0]])
assert np.array_equal(Ymed, YmedBook), "中央値フィルタ：教科書の解答と一致しない"
print(Ymed.astype(int))

教科書の解答にならって，局所領域を昇順に並べて 5 番目に下線を引いた列を作る（$y[1,0]$）。

In [ ]:
s10 = np.sort(win(1, 0).ravel())
print(s10.astype(int))
print(ordlist(s10, K), "→", f"{s10[K - 1]:g}")
assert ordlist(s10, K) == "0,0,0,9,\\underline{9},9,18,27,36"

## 最大値／最小値フィルタ
順序統計フィルタで $K=9$（最大値），$K=1$（最小値）とする（教科書 例題「最大値／最小値フィルタ」）。

In [ ]:
Ymax = ndimage.rank_filter(X, rank=8, size=3, mode="constant", cval=0)
Ymin = ndimage.rank_filter(X, rank=0, size=3, mode="constant", cval=0)
print(Ymax.astype(int))
print(Ymin.astype(int))
YmaxBook = np.array([[27, 27, 9, 9], [36, 36, 9, 9], [36, 36, 9, 9]])
YminBook = np.array([[0, 0, 0, 0], [0, 9, 9, 0], [0, 0, 0, 0]])
assert np.array_equal(Ymax, YmaxBook), "最大値フィルタ：教科書の解答と一致しない"
assert np.array_equal(Ymin, YminBook), "最小値フィルタ：教科書の解答と一致しない"

## 中央値フィルタの処理例（インパルス性ノイズ）
ノイズ密度 5% のごま塩ノイズを加え，平均値フィルタと中央値フィルタを比べる（PSNR の値はスライドとわずかに異なる）。

In [ ]:
Cs = random_noise(C, mode="s&p", amount=0.05, rng=1)
Csbox = ndimage.correlate(Cs, fbox, mode="nearest")
Csmed = ndimage.median_filter(Cs, size=3, mode="reflect")   # 周囲は鏡像で拡張
psnrS = [peak_signal_noise_ratio(C, v, data_range=1) for v in (Cs, Csbox, Csmed)]
print("psnrS =", np.round(psnrS, 1), "dB")
fig, axs = plt.subplots(2, 2, figsize=(7, 7.4), layout="constrained")
show(axs[0, 0], C, "原画像")
show(axs[0, 1], Cs, "雑音重畳（インパルス性）")
show(axs[1, 0], Csbox, "平均値フィルタ")
show(axs[1, 1], Csmed, "中央値フィルタ")
plt.show()

## 一次微分と二次微分（差分）
一定区間・ランプ・ステップを含む一次元信号 $x[n]$ に，一次差分 $x[n+1]-x[n]$（カーネル $(-1\ \ 1)^\top$）と二次差分 $x[n+1]-2x[n]+x[n-1]$（カーネル $(1\ \ {-2}\ \ 1)^\top$）を施す。

In [ ]:
x1 = np.array([6, 6, 6, 6, 5, 4, 3, 2, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6], dtype=float)
n1 = np.arange(len(x1))
d1 = np.r_[x1[1:] - x1[:-1], np.nan]                       # 一次差分（最後は未定義）
d2 = np.r_[np.nan, x1[2:] + x1[:-2] - 2 * x1[1:-1], np.nan]   # 二次差分（両端は未定義）

各段の題に差分の式（教科書の記法）を書き，入力の段にはランプとステップの位置を注記する。

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(5.6, 6.4), layout="constrained", sharex=True)
lbl = ["$x[n]$", "$x[n+1]-x[n]$", "$x[n+1]-2x[n]+x[n-1]$"]
yl = [(0, 7.5), (-2, 6), (-6, 6)]                           # 各段の縦軸の範囲
for ax, v, t, lim in zip(axs, (x1, d1, d2), lbl, yl):
    ok = ~np.isnan(v)
    ml, sl, bl = ax.stem(n1[ok], v[ok], basefmt="k-")
    plt.setp(ml, color=cmain, markersize=4)
    plt.setp(sl, color=cmain, linewidth=1)
    ax.set(title=t, xlim=(-0.5, n1[-1] + 0.5), xticks=np.arange(0, 19, 2), ylim=lim)
    ax.grid(True)
axs[0].text(6.6, 5.0, "ランプ", color=cwarm, ha="center")
axs[0].text(11.2, 3.5, "ステップ", color=cwarm, ha="center")
axs[2].set_xlabel("$n$")
plt.show()
print("d1 =", d1)
print("d2 =", d2)

## ラプラシアンフィルタ
4 近傍と 8 近傍のカーネル。8 近傍は教科書の一般形 $c\left((1-\alpha)\,\cdot\,+\alpha\,\cdot\right)$ で $\alpha=1/2$，$c=2$ とした場合に当たる。

In [ ]:
flap4 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
flapx = np.array([[1, 0, 1], [0, -4, 0], [1, 0, 1]])        # 対角方向の 4 近傍
flap8 = np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]])
alpha, c = 1 / 2, 2
assert np.array_equal(c * ((1 - alpha) * flap4 + alpha * flapx), flap8)

数値例の配列 $\mathsf{x}$ に 4 近傍ラプラシアンを施す（周囲は零値，教科書 例題「4近傍ラプラシアン」）。

In [ ]:
Ylap = ndimage.correlate(X, flap4, mode="constant")
print(Ylap.astype(int))
YlapBook = np.array([[-36, 0, -9, -18], [-45, 18, 0, -9], [-108, 18, -9, -18]])
assert np.array_equal(Ylap, YlapBook), "4近傍ラプラシアン：教科書の解答と一致しない"

$y[1,1]$ の積和を，係数と画素値の積を行ごとに並べて示す。

In [ ]:
e = prodexpr(flap4, win(1, 1))
print(e, "=", int(Ylap[1, 1]))
assert np.sum(flap4 * win(1, 1)) == Ylap[1, 1]
assert e == "0\\cdot18+1\\cdot9+0\\cdot9+1\\cdot27-4\\cdot9+1\\cdot9+0\\cdot36+1\\cdot9+0\\cdot9"

## ラプラシアンフィルタの処理例
参考資料のサンプル画像 msipimg04（石造りの建物）をグレースケールにして $256\times 256$ 画素に縮小し，4 近傍・8 近傍ラプラシアンを施す。暗い背景と平坦な地面に囲まれたアーチの輪郭が，変化部としてはっきり抽出される。出力は負の値を含むので，表示のため $0.5+ y$（バイアス処理）で示す。

In [ ]:
Mo = img_as_float(vie.msipimg(4, 256, "gray"))
L4 = ndimage.correlate(Mo, flap4.astype(float), mode="nearest")
L8 = ndimage.correlate(Mo, flap8.astype(float), mode="nearest")
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], Mo, "原画像")
show(axs[1], 0.5 + L4, "4 近傍")
show(axs[2], 0.5 + L8, "8 近傍")
plt.show()

## 高域強調（アンシャープマスク）
$\mathsf{y}=\mathsf{x}-\nabla^2\mathsf{x}$。カーネルは恒等変換からラプラシアンを引いたもの。

In [ ]:
fid = np.zeros((3, 3), dtype=int)
fid[1, 1] = 1                                  # 恒等変換のカーネル
fus4 = fid - flap4
fus8 = fid - flap8
print(fus4)
print(fus8)

MATLAB の `fspecial("unsharp",0)` は 4 近傍のカーネルに一致する。ここでは `fspecial` の定義式 $\frac{1}{\alpha+1}\begin{pmatrix}-\alpha&\alpha-1&-\alpha\\\alpha-1&\alpha+5&\alpha-1\\-\alpha&\alpha-1&-\alpha\end{pmatrix}$ で確かめる。

In [ ]:
def fspecial_unsharp(a):
    """fspecial("unsharp", a) の式でアンシャープマスクのカーネルを作る．"""
    return np.array([[-a, a - 1, -a], [a - 1, a + 5, a - 1], [-a, a - 1, -a]]) / (a + 1)


assert np.array_equal(fus4, fspecial_unsharp(0))

数値例の配列 $\mathsf{x}$ に 4 近傍アンシャープマスクを施す（教科書 例題「4近傍アンシャープマスク」）。入力から例題「4近傍ラプラシアン」の結果を引いても，カーネル `fus4` で近傍処理しても同じ。

In [ ]:
Yus = ndimage.correlate(X, fus4, mode="constant")
print(Yus.astype(int))
assert np.array_equal(Yus, X - Ylap)
YusBook = np.array([[54, 9, 18, 27], [72, -9, 9, 18], [144, -9, 18, 27]])
assert np.array_equal(Yus, YusBook), "4近傍アンシャープマスク：教科書の解答と一致しない"

画素毎に「中央の 5 倍から上・左・右・下の 4 近傍の和を引く」形で $y[1,1]$ を示す。

In [ ]:
W = win(1, 1)
e = f"{fus4[1, 1]}\\cdot{numtex(W[1, 1])}" + "-(" + sumexpr([W[0, 1], W[1, 0], W[1, 2], W[2, 1]]) + ")"
print(e, "=", int(Yus[1, 1]))
assert e == "5\\cdot9-(9+27+9+9)"

In [ ]:
U4 = ndimage.correlate(Mo, fus4.astype(float), mode="nearest")
U8 = ndimage.correlate(Mo, fus8.astype(float), mode="nearest")
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], Mo, "原画像")
show(axs[1], U4, "4 近傍")
show(axs[2], U8, "8 近傍")
plt.show()

## プレウィットフィルタの数値例
垂直方向 $f_\mathrm{v}$ と水平方向 $f_\mathrm{h}$ のカーネルで差分画像 $\mathsf{x}_\mathrm{v}$，$\mathsf{x}_\mathrm{h}$ を求める（教科書 例題「プレウィットフィルタ」）。

In [ ]:
fpv = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])
fph = fpv.T
Xv = ndimage.correlate(X, fpv, mode="constant")
Xh = ndimage.correlate(X, fph, mode="constant")
print(Xv.astype(int))
print(Xh.astype(int))
XvBook = np.array([[36, 45, 27, 18], [18, 18, 0, 0], [-36, -45, -27, -18]])
XhBook = np.array([[18, -27, 0, -18], [27, -54, 0, -27], [18, -45, 0, -18]])
assert np.array_equal(Xv, XvBook) and np.array_equal(Xh, XhBook), "プレウィットフィルタ：教科書の解答と一致しない"

$x_\mathrm{v}[1,1]$ は「下の行の和 − 上の行の和」，$x_\mathrm{h}[1,1]$ は「右の列の和 − 左の列の和」。

In [ ]:
W = win(1, 1)
ev = "-(" + sumexpr(W[0, :]) + ")+(" + sumexpr(W[2, :]) + ")"
eh = "-(" + sumexpr(W[:, 0]) + ")+(" + sumexpr(W[:, 2]) + ")"
print(ev, "=", int(Xv[1, 1]))
print(eh, "=", int(Xh[1, 1]))
assert W[2, :].sum() - W[0, :].sum() == Xv[1, 1] and W[:, 2].sum() - W[:, 0].sum() == Xh[1, 1]
assert ev == "-(18+9+9)+(36+9+9)" and eh == "-(18+27+36)+(9+9+9)"

勾配の大きさ $y_\mathrm{mag}[\boldsymbol{n}]=\sqrt{x_\mathrm{v}^2[\boldsymbol{n}]+x_\mathrm{h}^2[\boldsymbol{n}]}$（教科書 例題「勾配の大きさ」，小数第 2 位まで）。

In [ ]:
Ymag = np.sqrt(Xv**2 + Xh**2)
np.set_printoptions(precision=2, suppress=True)
print(Ymag)
YmagBook = np.array([[40.25, 52.48, 27.00, 25.46], [32.45, 56.92, 0.00, 27.00], [40.25, 63.64, 27.00, 25.46]])
r2 = np.round(Ymag, 2)                         # 小数第 2 位まで
assert np.max(np.abs(r2 - YmagBook)) < 1e-12, "勾配の大きさ：教科書の解答と一致しない"
em = f"\\sqrt{{{numtex(Xv[1, 1])}^2+{numtex(Xh[1, 1])}^2}}"
print(em, f"= {Ymag[1, 1]:.2f}")
assert em == "\\sqrt{18^2+(-54)^2}"

## 勾配フィルタの処理例
参考資料のサンプル画像 msipimg03（マカロン）をグレースケールにして $256\times 256$ 画素に縮小し，プレウィットとソーベルを施して勾配の大きさを表示する（最大値で正規化）。規則的に並ぶ丸い菓子の輪郭が抽出される。

In [ ]:
Co = img_as_float(vie.msipimg(3, 256, "gray"))
fsv = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])
fsh = fsv.T
Gp = np.hypot(ndimage.correlate(Co, fpv.astype(float), mode="nearest"),
              ndimage.correlate(Co, fph.astype(float), mode="nearest"))
Gs = np.hypot(ndimage.correlate(Co, fsv.astype(float), mode="nearest"),
              ndimage.correlate(Co, fsh.astype(float), mode="nearest"))
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], Co, "原画像")
show(axs[1], Gp / Gp.max(), "プレウィット")
show(axs[2], Gs / Gs.max(), "ソーベル")
plt.show()

## 教科書の例題との照合（まとめ）
ここまでの `assert` をすべて通過した。各例題の結果を一覧にする。

In [ ]:
agree = {
    "矩形フィルタ": np.array_equal(Ybox, YboxBook),
    "ガウシアンフィルタ（σ=0.5，σ≃0.849）":
        max(np.abs(fg05 - fg05Book).max(), np.abs(fg - fgBook).max()) < 5e-5,   # 教科書は小数第 4 位まで
    "4近傍ラプラシアン": np.array_equal(Ylap, YlapBook),
    "4近傍アンシャープマスク": np.array_equal(Yus, YusBook),
    "プレウィットフィルタ": np.array_equal(Xv, XvBook) and np.array_equal(Xh, XhBook),
    "勾配の大きさ": np.max(np.abs(r2 - YmagBook)) < 1e-12,                     # 教科書は小数第 2 位まで
    "中央値フィルタ": np.array_equal(Ymed, YmedBook),
    "最大値／最小値フィルタ": np.array_equal(Ymax, YmaxBook) and np.array_equal(Ymin, YminBook),
}
for name, ok in agree.items():
    print(f"{'一致' if ok else '不一致'}：{name}")

## 二次元インパルス信号
$\delta[n_1,n_2]=\delta[n_1]\,\delta[n_2]$ は原点だけが 1 の配列（教科書の記法：次元の添え字は 1 始まり）。

In [ ]:
n1g, n2g = np.meshgrid(np.arange(-3, 4), np.arange(-3, 4))
d2d = ((n1g == 0) & (n2g == 0)).astype(float)
fig = plt.figure(figsize=(5, 4), layout="constrained")
ax = fig.add_subplot(projection="3d")
ml, sl, bl = ax.stem(n1g.ravel(), n2g.ravel(), d2d.ravel(), basefmt=" ")
plt.setp(ml, color=cmain, markersize=4)
plt.setp(sl, color=cmain, linewidth=1.2)
ax.set(xlabel="$n_1$", ylabel="$n_2$", zlabel="$\\delta[n_1,n_2]$", zticks=[0, 0.5, 1])
ax.view_init(elev=30, azim=-125)
plt.show()

## 畳み込みの数値例
一次元の入力 $(x[n])_n=(1\ 2\ 3\ 2)^\top\in\mathbb{R}^{\{0,1,2,3\}}$ とインパルス応答 $(h[n])_n=(1\ 2\ 1)^\top\in\mathbb{R}^{\{0,1,2\}}$ の畳み込み $y[n]=\sum_{m} x[m]h[n-m]$（教科書 定理 4.1 の記法）。入力をインパルスの重み付け和とみなし，各インパルスの応答を足し合わせる。

In [ ]:
xc = np.array([1, 2, 3, 2])
hc = np.array([1, 2, 1])
yc = np.convolve(xc, hc)
print("y =", yc)
parts = np.zeros((len(xc), len(yc)), dtype=int)
for k in range(len(xc)):
    parts[k, k:k + len(hc)] = xc[k] * hc       # x[k] h[n-k]
print(parts)
assert np.array_equal(parts.sum(axis=0), yc)
assert np.array_equal(yc, [1, 4, 8, 10, 7, 2])  # スライドの値

スライドの表（行：$x[k]h[n-k]$，列：$n$）の形で示す。インパルス応答の台の外は空欄にする。

In [ ]:
print(f"{'n':>10} |" + "".join(f"{n:>4}" for n in range(len(yc))))
for k in range(len(xc)):
    lab = "x[0]h[n]" if k == 0 else f"x[{k}]h[n-{k}]"
    cells = ["" if not (k <= n < k + len(hc)) else str(parts[k, n]) for n in range(len(yc))]
    print(f"{lab:>10} |" + "".join(f"{s:>4}" for s in cells))
print(f"{'y[n]':>10} |" + "".join(f"{v:>4}" for v in yc))

## まとめ
- 線形フィルタは局所的な積和演算。平均値・加重平均値フィルタは平滑化，ラプラシアン・アンシャープマスクは先鋭化
- 中央値フィルタは非線形で，インパルス性ノイズに強い
- 勾配フィルタは変化の強さと向きを求める
- 線形シフト不変システムはインパルス応答との畳み込みで表される

© Copyright, Shogo MURAMATSU, All rights reserved.